# Семинар 8: Сети
---
## Часть 1: Граф как инструмент гуманитарного исследования

### Что такое граф?

**Граф (Network / Graph)** — математическая структура, состоящая из **объектов** и **отношений** между ними. В зависимости от предметной области объектами могут быть персонажи пьесы, исторические деятели, города, научные статьи.

Формально граф — это $G = (V, E)$, где $V$ — множество вершин (vertices), $E$ — множество рёбер (edges). 

### Узлы и рёбра

| Термин | Синонимы | Пример |
|--------|----------|-------------------------|
| **Узел / Вершина** | Node, Vertex | Конкретный человек: Карл V, Максимилиан I |
| **Ребро / Связь** | Edge, Link | Отношение «является отцом» |

### Типология графов

**Неориентированный граф (Undirected Graph)** — связи симметричны. Если A связан с B, то и B связан с A. Пример: брак, совместное присутствие на сцене, факт знакомства. В библиотеке `networkx`: `nx.Graph()`.

![Неориентированный граф](https://upload.wikimedia.org/wikipedia/commons/thumb/b/bf/Undirected.svg/250px-Undirected.svg.png)

**Ориентированный граф (Directed Graph / DiGraph)** — связи имеют направление. Цитирование (А цитирует Б, но не наоборот), генеалогия (А — отец Б), вассальная зависимость. В библиотеке `networkx`: `nx.DiGraph()`. Наш датасет — ориентированный: стрелка идёт от отца к сыну.

![Ориентированный граф](https://upload.wikimedia.org/wikipedia/commons/thumb/a/a2/Directed.svg/330px-Directed.svg.png)

**Взвешенный граф (Weighted Graph)** — у каждого ребра есть числовое значение (вес). В сетевом анализе драмы весом может служить количество реплик между персонажами; в библиографии — число совместных публикаций.

![Взвешенный граф](https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/Network_flow_9_vertices_blank.svg/960px-Network_flow_9_vertices_blank.svg.png)

### Метрики

**Степень узла (Degree)** — количество рёбер, подходящих к узлу. В ориентированном графе:
- **In-Degree** — входящие связи 
- **Out-Degree** — исходящие связи 

**Плотность сети (Density)** — отношение числа существующих рёбер к максимально возможному. Значение от 0 до 1: чем ближе к 1, тем теснее переплетена сеть.

$\text{Density} = \frac{|E|}{|V| \cdot (|V|-1)}$

Для ориентированного графа делитель $|V|(|V|-1)$, для неориентированного — $\frac{|V|(|V|-1)}{2}$.

Центральность (Centrality) — это семейство мер важности узла. Разные метрики отвечают на разные вопросы:

**Degree Centrality** — нормированная степень узла: насколько узел популярен по прямым связям. Просто делим степень на $(n-1)$, чтобы получить значение от 0 до 1, независимо от размера сети.

**Betweenness Centrality (центральность по посредничеству)** — как часто данный узел оказывается на кратчайшем пути между всеми другими парами узлов. В истории так выявляют ключевых дипломатов, через которых шли все переговоры, или персонажей, связывающих разные социальные группы.

$$C_B(v) = \sum_{s \neq v \neq t} \frac{\sigma_{st}(v)}{\sigma_{st}}$$

где $\sigma_{st}$ — общее число кратчайших путей от $s$ до $t$, а $\sigma_{st}(v)$ — число тех из них, что проходят через $v$.

---

Для работы нам понадобятся две новые библиотеки:
- `networkx` — математика графов: создание, анализ, метрики
- `pyvis` — интерактивная HTML-визуализация графов

In [2]:
#!pip install beautifulsoup4 lxml
from bs4 import BeautifulSoup
from collections import defaultdict
import pandas as pd

#!pip install pyvis
#!pip install networkx
import networkx as nx
from pyvis.network import Network

---
## Часть 2: NetworkX 

Прежде чем работать с реальными данными, разберём логику `networkx` на маленьком примере. Представим, что нам известны пять человек и несколько отношений между ними.

Сначала построим **неориентированный граф** — например, «кто с кем знаком».

In [12]:
# Создаём неориентированный граф
G = nx.Graph()

G

In [13]:
# Добавляем узлы вручную
G.add_node("Анна")
G.add_nodes_from(["Борис", "Вера", "Глеб", "Дина"])

In [14]:
# Добавляем рёбра — связи между узлами
G.add_edge("Анна", "Борис")
G.add_edge("Анна", "Вера")
G.add_edge("Анна", "Глеб")
G.add_edge("Борис", "Глеб")
G.add_edge("Глеб", "Дина")
G.add_edge("Вера", "Дина")

In [15]:
print("Узлы:", list(G.nodes()))
print("Рёбра:", list(G.edges()))
print("Число узлов:", G.number_of_nodes())
print("Число рёбер:", G.number_of_edges())

Узлы: ['Анна', 'Борис', 'Вера', 'Глеб', 'Дина']
Рёбра: [('Анна', 'Борис'), ('Анна', 'Вера'), ('Анна', 'Глеб'), ('Борис', 'Глеб'), ('Вера', 'Дина'), ('Глеб', 'Дина')]
Число узлов: 5
Число рёбер: 6


Теперь вычислим метрики. 

In [16]:
# Степень каждого узла
print("Степени узлов:")
for node, degree in G.degree():
    print(f"{node}: {degree}")

print()

Степени узлов:
Анна: 3
Борис: 2
Вера: 2
Глеб: 3
Дина: 2



In [19]:
# Плотность сети
density = nx.density(G)
print(f"Плотность сети: {density:.1f}")
print(f"{density*100:.1f}% от всех возможных связей существует")

Плотность сети: 0.6
60.0% от всех возможных связей существует


In [21]:
# Метрики центральности
degree_centrality = nx.degree_centrality(G)
betweenness_centrality = nx.betweenness_centrality(G)

print(degree_centrality)

{'Анна': 0.75, 'Борис': 0.5, 'Вера': 0.5, 'Глеб': 0.75, 'Дина': 0.5}


In [24]:
print("Degree Centrality (нормированная степень):")
for node, val in sorted(degree_centrality.items(), key=lambda x: x[1], reverse=True):
    print(f"  {node}: {val:.2f}")

print()
print("Betweenness Centrality (посредничество):")
for node, val in sorted(betweenness_centrality.items(), key=lambda x: x[1], reverse=True):
    print(f"  {node}: {val:.2f}")

Degree Centrality (нормированная степень):
  Анна: 0.75
  Глеб: 0.75
  Борис: 0.50
  Вера: 0.50
  Дина: 0.50

Betweenness Centrality (посредничество):
  Анна: 0.25
  Глеб: 0.25
  Вера: 0.08
  Дина: 0.08
  Борис: 0.00


Теперь построим **ориентированный граф** — связи имеют направление. Используем пример с генеалогией: стрелка идёт от отца к сыну.

Обратите внимание: единственное отличие в коде — `nx.DiGraph()` вместо `nx.Graph()`.

In [25]:
# Создаём ориентированный граф
DG = nx.DiGraph()

# Добавляем рёбра: (отец, сын) — стрелка от отца к сыну
DG.add_edge("Дед", "Отец")
DG.add_edge("Дед", "Дядя")
DG.add_edge("Дед", "Тётя")
DG.add_edge("Дядя", "Кузина")
DG.add_edge("Отец", "Я")
DG.add_edge("Отец", "Брат")
DG.add_edge("Отец", "Сестра")

print("Узлы:", list(DG.nodes()))
print("Рёбра:", list(DG.edges()))
print()

Узлы: ['Дед', 'Отец', 'Дядя', 'Тётя', 'Кузина', 'Я', 'Брат', 'Сестра']
Рёбра: [('Дед', 'Отец'), ('Дед', 'Дядя'), ('Дед', 'Тётя'), ('Отец', 'Я'), ('Отец', 'Брат'), ('Отец', 'Сестра'), ('Дядя', 'Кузина')]



In [27]:
# В ориентированном графе степень разделяется на in и out
print("In-degree (входящие — сколько сыновей у узла):")
for node, deg in DG.in_degree():
    print(f"  {node}: {deg}")

print()
print("Out-degree (исходящие — есть ли у узла отец):")
for node, deg in DG.out_degree():
    print(f"  {node}: {deg}")

In-degree (входящие — сколько сыновей у узла):
  Дед: 0
  Отец: 1
  Дядя: 1
  Тётя: 1
  Кузина: 1
  Я: 1
  Брат: 1
  Сестра: 1

Out-degree (исходящие — есть ли у узла отец):
  Дед: 3
  Отец: 3
  Дядя: 1
  Тётя: 0
  Кузина: 0
  Я: 0
  Брат: 0
  Сестра: 0


Метрики центральности для ориентированного графа можно считать методами неориентированных графов и специальными `in_` / `out_` методами. Результаты будут разными:

In [28]:
print(f"Метод для неориентированного графа: {nx.degree_centrality(DG)}")

print(f"Метод in_: {nx.in_degree_centrality(DG)}")

print(f"Метод out_: {nx.out_degree_centrality(DG)}")

Метод для неориентированного графа: {'Дед': 0.42857142857142855, 'Отец': 0.5714285714285714, 'Дядя': 0.2857142857142857, 'Тётя': 0.14285714285714285, 'Кузина': 0.14285714285714285, 'Я': 0.14285714285714285, 'Брат': 0.14285714285714285, 'Сестра': 0.14285714285714285}
Метод in_: {'Дед': 0.0, 'Отец': 0.14285714285714285, 'Дядя': 0.14285714285714285, 'Тётя': 0.14285714285714285, 'Кузина': 0.14285714285714285, 'Я': 0.14285714285714285, 'Брат': 0.14285714285714285, 'Сестра': 0.14285714285714285}
Метод out_: {'Дед': 0.42857142857142855, 'Отец': 0.42857142857142855, 'Дядя': 0.14285714285714285, 'Тётя': 0.0, 'Кузина': 0.0, 'Я': 0.0, 'Брат': 0.0, 'Сестра': 0.0}


### Атрибуты узлов и рёбер в NetworkX

Каждый узел и каждое ребро в NetworkX — это словарь. В него можно записать любую информацию: числа, строки, списки.

In [29]:
# Атрибуты можно задать сразу при добавлении узла или ребра
DG.add_node("Дед", surname="Васильев", birth_year=1941)
DG.add_edge("Дед", "Отец", relation="father_of", weight=1)

# Атрибуты хранятся как обычный словарь
print("Атрибуты узла:", DG.nodes["Дед"])
print("Атрибуты ребра:", DG.edges["Дед", "Отец"])

Атрибуты узла: {'surname': 'Васильев', 'birth_year': 1941}
Атрибуты ребра: {'relation': 'father_of', 'weight': 1}


In [30]:
# Атрибуты можно дописать в любой момент — как в обычный словарь
DG.nodes["Отец"]["birth_year"] = 1962
DG.nodes["Брат"]["birth_year"] = 1996

print(DG.nodes["Отец"])
print(DG.nodes["Брат"])

{'birth_year': 1962}
{'birth_year': 1996}


In [31]:
print(DG.adj) #AdjacencyView — специальный объект, который представляет структуру связей (ребер) графа в виде словаря словарей

{'Дед': {'Отец': {'relation': 'father_of', 'weight': 1}, 'Дядя': {}, 'Тётя': {}}, 'Отец': {'Я': {}, 'Брат': {}, 'Сестра': {}}, 'Дядя': {'Кузина': {}}, 'Тётя': {}, 'Кузина': {}, 'Я': {}, 'Брат': {}, 'Сестра': {}}


In [32]:
DG.nodes

NodeView(('Дед', 'Отец', 'Дядя', 'Тётя', 'Кузина', 'Я', 'Брат', 'Сестра'))

In [33]:
DG.out_edges

OutEdgeView([('Дед', 'Отец'), ('Дед', 'Дядя'), ('Дед', 'Тётя'), ('Отец', 'Я'), ('Отец', 'Брат'), ('Отец', 'Сестра'), ('Дядя', 'Кузина')])

In [34]:
DG.in_edges

InEdgeView([('Дед', 'Отец'), ('Дед', 'Дядя'), ('Дед', 'Тётя'), ('Дядя', 'Кузина'), ('Отец', 'Я'), ('Отец', 'Брат'), ('Отец', 'Сестра')])

In [35]:
# data=True — просим вернуть не только имена узлов, но и их атрибуты
for node, attrs in DG.nodes(data=True):
    print(f"{node}: {attrs}")

Дед: {'surname': 'Васильев', 'birth_year': 1941}
Отец: {'birth_year': 1962}
Дядя: {}
Тётя: {}
Кузина: {}
Я: {}
Брат: {'birth_year': 1996}
Сестра: {}


In [36]:
# Для рёбер аналогично: три значения при data=True — откуда, куда, атрибуты
for source, target, attrs in DG.edges(data=True):
    print(f"{source} -> {target}: {attrs}")

Дед -> Отец: {'relation': 'father_of', 'weight': 1}
Дед -> Дядя: {}
Дед -> Тётя: {}
Отец -> Я: {}
Отец -> Брат: {}
Отец -> Сестра: {}
Дядя -> Кузина: {}


Запись метрик в атрибуты узлов — стандартный паттерн работы с NetworkX. Сначала вычисляем метрику (получаем словарь `{узел: значение}`), затем записываем результат обратно в граф. Тогда вся информация об узле хранится в одном месте, и `PyVis` сможет её прочитать при визуализации.

In [37]:
# Вычисляем метрику — получаем словарь
centrality = nx.in_degree_centrality(DG)
print("Словарь метрик:", centrality)

# Записываем каждое значение как атрибут узла
for node in DG.nodes():
    DG.nodes[node]["centrality"] = centrality[node]

# Проверяем: теперь centrality хранится прямо внутри графа
for node, attrs in DG.nodes(data=True):
    print(f"{node}: {attrs}")

Словарь метрик: {'Дед': 0.0, 'Отец': 0.14285714285714285, 'Дядя': 0.14285714285714285, 'Тётя': 0.14285714285714285, 'Кузина': 0.14285714285714285, 'Я': 0.14285714285714285, 'Брат': 0.14285714285714285, 'Сестра': 0.14285714285714285}
Дед: {'surname': 'Васильев', 'birth_year': 1941, 'centrality': 0.0}
Отец: {'birth_year': 1962, 'centrality': 0.14285714285714285}
Дядя: {'centrality': 0.14285714285714285}
Тётя: {'centrality': 0.14285714285714285}
Кузина: {'centrality': 0.14285714285714285}
Я: {'centrality': 0.14285714285714285}
Брат: {'birth_year': 1996, 'centrality': 0.14285714285714285}
Сестра: {'centrality': 0.14285714285714285}


---
## Часть 3. PyVis: интерактивная визуализация

`pyvis` работает по следующей логике: мы создаём объект, передаём в него данные, и на выходе получаем HTML-файл с интерактивной визуализацией.

Ключевое удобство: `pyvis` умеет напрямую импортировать граф из `networkx` — не нужно пересобирать всё заново.

In [ ]:
# Создаём объект Network
# bgcolor — цвет фона, font_color — цвет подписей
net = Network(
    height="500px",        # высота холста в пикселях
    width="100%",           # ширина — 100% от ширины ячейки в тетрадке
    bgcolor="#1a1a2e",    # цвет фона в формате HEX (тёмно-синий)
    font_color="white",     # цвет подписей у узлов
    directed=True           # граф ориентированный: рёбра будут отображаться со стрелками
)

# Импортируем граф из networkx
net.from_nx(DG)

# Включаем физику — узлы будут отталкиваться и находить равновесие
net.toggle_physics(True)

#net.show_buttons(filter_=['physics'])

# Сохраняем как HTML
net.write_html("toy_graph.html")

**Что такое «физика»?** 

`pyvis` использует физический симулятор: узлы ведут себя как заряженные частицы, а рёбра — как пружины. При запуске они разлетаются от центра и постепенно находят устойчивое положение, где силы уравновешены, что позволяет автоматически получить читаемую раскладку без ручного указания координат.

### Параметры физического симулятора

PyVis использует библиотеку vis.js, и по умолчанию применяет алгоритм **Barnes-Hut** — физическую модель, где узлы отталкиваются друг от друга как заряженные частицы, а рёбра притягивают соединённые узлы как пружины.

Самые важные параметры:

**`gravitationalConstant`** (по умолчанию: -2000)

Сила отталкивания между всеми узлами. Чем больше отрицательное число по модулю, тем сильнее узлы разлетаются от центра. При значении близком к 0 узлы слипаются в кучу.

**Когда менять:** если узлы слишком плотно сгрудились — увеличьте отрицательное значение.

**`springLength`** (по умолчанию: 95) и **`springConstant`** (по умолчанию: 0.04)

`springLength` — желаемая длина ребра в пикселях: на каком расстоянии симулятор будет держать два связанных узла.

`springConstant` — жёсткость пружины: насколько настойчиво ребро тянет узлы к этому расстоянию. Высокое значение — граф собирается туго, низкое — узлы гуляют свободнее.

**Когда менять:** если связанные узлы расположены слишком далеко или слишком близко друг к другу.

**`damping`** (по умолчанию: 0.09)

Затухание — насколько быстро узлы успокаиваются и перестают двигаться. Значение от 0 до 1: при 0 узлы колышутся бесконечно, при 1 — замирают мгновенно.

**Когда менять:** если анимация слишком долго не останавливается — увеличьте значение.

Значения параметров по умолчанию работают для большинства небольших графов. Панель `show_buttons(filter_=['physics'])` нужна именно для того, чтобы подобрать нужные значения вручную в браузере, а потом зафиксировать их через `set_options()` в коде.

In [ ]:
# Параметры физики можно задать прямо в коде, не используя панель в браузере
# Используйте, когда нужно зафиксировать конкретный вид графа для публикации

net.set_options("""
const options = {
  "physics": {
    "minVelocity": 0.75,
    "solver": "repulsion"
  }
}
""")

#net.write_html("toy_graph.html")

---
## Часть 4. Кастомизация: связываем метрики с визуализацией

Сделаем так, чтобы размер узла соответствовал его центральности. Чем важнее предок — тем крупнее его кружок на графе.

Для этого нужно передать метрики в атрибуты узлов графа `networkx` **до** того, как мы передадим его в `pyvis`.

### Атрибуты узлов в PyVis

Узлам графа можно задавать визуальные свойства через атрибуты.
PyVis автоматически читает их при импорте из networkx.

Основные атрибуты:
- `size` — размер кружка на холсте (число, обычно от 5 до 50)
- `title` — текст всплывающей подсказки при наведении курсора
- `color` — цвет узла (HEX-строка или название цвета)
- `label` — подпись, которая видна прямо на холсте (по умолчанию — имя узла)

In [56]:
# Начнём с самого простого: зададим одинаковый цвет всем узлам
for node in DG.nodes():
    DG.nodes[node]["color"] = "#e8a838"

# Проверяем: атрибуты хранятся прямо в объекте графа как словарь
print(DG.nodes["Отец"])

{'birth_year': 1962, 'centrality': 0.14285714285714285, 'size': 10, 'color': '#e8a838'}


In [57]:
# size задаём не константой, а формулой: базовый размер + надбавка за центральность
# без базового значения узлы с нулевой метрикой стали бы невидимы
for node in DG.nodes():
    DG.nodes[node]["size"] = 15 + centrality[node] * 60

# Смотрим, что получилось у конкретного узла
print("Отец:", DG.nodes["Отец"]["size"])
print("Дед:", DG.nodes["Дед"]["size"])

Отец: 23.57142857142857
Дед: 15.0


In [58]:
# Вычисляем дополнительную метрику betweenness на нашем игрушечном DiGraph

betweenness = nx.betweenness_centrality(DG)

In [59]:
# title — это HTML-строка, которая появится при наведении курсора
# \n работает как перенос строки внутри подсказки
for node in DG.nodes():
    DG.nodes[node]["title"] = (
        f"{node}\n"
        f"In-degree centrality: {centrality[node]:.3f}\n"
        f"Betweenness: {betweenness[node]:.3f}"
    )

print(DG.nodes["Отец"]["title"])

Отец
In-degree centrality: 0.143
Betweenness: 0.071


In [60]:
# Создаём новую сеть с обновлёнными атрибутами
net2 = Network(
    height="800px",
    width="100%",
    bgcolor="#a1a1c2",
    font_color="white",
    directed=True
)
net2.from_nx(DG)

# Добавляем интерактивное меню для настройки физики прямо в браузере
net2.show_buttons(filter_=["physics"])

net2.write_html("toy_graph_styled.html")

Наведите курсор на любой узел — должна появиться всплывающая подсказка со значениями метрик. Воспользуйтесь панелью «Physics» в правой части, чтобы изменить поведение симулятора без перезапуска кода.

## Часть 5. Персинг разметки TEI
*TEI* (Text Encoding Initiative) — это международный стандарт для представления текстов в машиночитаемом виде. Это язык разметки текстов, основанный на XML, который позволяет исследователям единообразно описывать структуру и содержание документов.

С помощью TEI вы можете явно выделить не только заголовки и абзацы, но и:
- имена персонажей (и добавить к ним атрибуты),
- географические названия,
- даты,
- исправления в рукописи,
- слова на иностранных языках,
- тип рифмы и тд.

Сегодня мы будем учиться парсить TEI на тексте драмы А.С. Пушкина "Борис Годунов". Размеченные тексты других авторов см. на сайте проекта [DraCor](https://dracor.org).

In [3]:
with open('rus000042-pushkin-boris-godunov.tei.xml', 'r', encoding='utf-8') as f:
    content = f.read()

soup = BeautifulSoup(content, 'xml')

In [4]:
scenes = soup.find_all('div', {'type': 'scene'})

scene_sp = defaultdict(set)
mapping = dict()

for id, scene in enumerate(scenes, 1):
    speeches = scene.find_all('sp')
    
    for sp in speeches:
        speaker = sp.get('who')
        scene_sp[id].add(speaker)
        mapping[speaker] = sp.find("speaker").text

    if id > 4: 
        continue
    else:
        print(f"Сцена {id}: {len(scene_sp[id])} персонажей")
        print(f"  {', '.join(sorted(scene_sp[id]))}")

print("\n\nМаппинг имён персонажей и их id:")
list(mapping.items())[:5]


Сцена 1: 2 персонажей
  #shujskij, #vorotynskij
Сцена 2: 5 персонажей
  #drugoj_iz_naroda_1, #narod, #odin_iz_naroda_1, #schelkalov, #tretij_iz_naroda_1
Сцена 3: 4 персонажей
  #bojare, #shujskij, #tsar_boris, #vorotynskij
Сцена 4: 2 персонажей
  #grigorij_dimitrij_lzhedimitrij_samozvanets, #pimen


Маппинг имён персонажей и их id:


[('#vorotynskij', 'Воротынский'),
 ('#shujskij', 'Князь Шуйский'),
 ('#odin_iz_naroda_1', 'Один'),
 ('#drugoj_iz_naroda_1', 'Другой'),
 ('#tretij_iz_naroda_1', 'Третий')]

In [ ]:
pairs = []

for scene, characters in scene_sp.items():
    char_list = list(characters)
    for i in range(len(char_list)):
        for j in range(i + 1, len(char_list)):
            char1, char2 = sorted([char_list[i], char_list[j]])
            pairs.append((char1, char2))


# Превращаем в DataFrame
df = pd.DataFrame(pairs, columns=['char1', 'char2'])

# Подсчитываем веса (сколько раз встречается каждая пара)
df = df.groupby(['char1', 'char2']).size().reset_index(name='weight')
df = df.sort_values('weight', ascending=False).reset_index(drop=True)

df = df[~df['char1'].str.contains('drugoj')]
df = df[~df['char2'].str.contains('drugoj')]

# теперь добавим персонажам настоящие имена
for id, row in df.iterrows():
    df.loc[id, "char1"] = mapping[row["char1"]]
    df.loc[id, "char2"] = mapping[row["char2"]]


df.head()

,char1,char2,weight
0,Князь Шуйский,Царь,3
1,Бояре,Царь,3
2,Самозванец,Все,2
3,Басманов,Царь,2
4,Пушкин,Самозванец,2


In [71]:
# Создаём граф NetworkX
G = nx.from_pandas_edgelist(df, 'char1', 'char2', 'weight')

degree_centrality = nx.degree_centrality(G)           
betweenness_centrality = nx.betweenness_centrality(G) 

# Задаём атрибуты для каждого узла
for node in G.nodes():
    G.nodes[node]["title"] = (
        f"{node}\n"
        f"Degree centrality: {degree_centrality[node]:.1f}\n"
        f"Betweenness: {betweenness_centrality[node]:.1f}"
    )
    G.nodes[node]["size"] = 9 + degree_centrality[node] * 40
    
net = Network(
    height="700px",
    width="100%",
    bgcolor="#ffffff",
    font_color="black"
)

# Импортируем граф из networkx
net.from_nx(G)

for node in net.nodes:
    centrality = degree_centrality.get(node['label'], 0)
    node['font'] = {'size': 20 + centrality * 60}


# Включаем физику
#net.toggle_physics(True)

#net.show_buttons(filter_=['physics'])

net.set_options("""
{
  "physics": {
    "barnesHut": {
      "gravitationalConstant": -7350,
      "avoidOverlap": 0.53
    },
    "minVelocity": 10,
    "maxVelocity": 10,
    "damping": 0.9
  }
}
""")

net.write_html("drama.html")